# 👑 DropQueen
## Notebook 4: Flask API
**Project:** AI-Powered Product Demand & Sales Forecasting Engine for TikTok Shop & Amazon  
**Author:** Chastity Lewis  
**Course:** CISC 610 — DevOps and MLOps | Mercy University | Spring 2026  

---

### 📌 Notebook Goals
1. Install dependencies and set up the Flask environment in Colab
2. Upload the saved `.pkl` models from Notebook 3
3. Load and verify both ML models
4. Build a Flask REST API with the following endpoints:
   - `GET  /health` — Health check
   - `POST /predict/demand` — Predict demand spike for a product
   - `POST /predict/trend` — Predict TikTok engagement level
   - `GET  /products/top` — Get top predicted demand products
   - `GET  /models/versions` — List model metadata
5. Test all API endpoints using `requests`
6. Save the complete `app.py` file for Docker deployment

---

In [4]:
!pip install flask flask-ngrok pyngrok joblib scikit-learn pandas numpy --quiet

## Step 1: Install & Import Libraries

In [5]:
!pip install flask pyngrok joblib scikit-learn pandas numpy --quiet

import os
import json
import threading
import requests
import joblib
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

from flask import Flask, request, jsonify
from datetime import datetime

print('✅ Libraries loaded!')

✅ Libraries loaded!


## Step 2: Upload Saved Models from Notebook 3

In [6]:
from google.colab import files
import io

print('📂 Upload your saved model files from Notebook 3:')
print('   - demand_model.pkl')
print('   - engagement_model.pkl')
print('   - model_config.json')
print()

uploaded = files.upload()

print(f'\n✅ Uploaded files: {list(uploaded.keys())}')

📂 Upload your saved model files from Notebook 3:
   - demand_model.pkl
   - engagement_model.pkl
   - model_config.json



Saving model_config.json to model_config (1).json
Saving demand_model.pkl to demand_model (2).pkl
Saving engagement_model.pkl to engagement_model (2).pkl

✅ Uploaded files: ['model_config (1).json', 'demand_model (2).pkl', 'engagement_model (2).pkl']


## Step 3: Load & Verify Models

In [7]:
# Load models
demand_model     = joblib.load('demand_model.pkl')
engagement_model = joblib.load('engagement_model.pkl')

with open('model_config.json', 'r') as f:
    model_config = json.load(f)

DEMAND_FEATURES = model_config['demand_features']
TIKTOK_FEATURES = model_config['tiktok_features']

print('✅ Models loaded successfully!')
print(f'\n🌲 Demand Model:     {type(demand_model).__name__}')
print(f'   Features:        {DEMAND_FEATURES}')
print(f'   Accuracy:        {model_config["demand_accuracy"]}')
print(f'   F1-Score:        {model_config["demand_f1"]}')
print(f'\n📊 Engagement Model: {type(engagement_model).__name__}')
print(f'   Features:        {TIKTOK_FEATURES}')
print(f'   Accuracy:        {model_config["tiktok_accuracy"]}')
print(f'   F1-Score:        {model_config["tiktok_f1"]}')

✅ Models loaded successfully!

🌲 Demand Model:     RandomForestClassifier
   Features:        ['weekly_reviews', 'avg_weekly_rating', 'rolling_avg_3w', 'week_sin', 'week_cos', 'high_rating_week', 'product_encoded']
   Accuracy:        1.0
   F1-Score:        1.0

📊 Engagement Model: LogisticRegression
   Features:        ['Likes', 'Comments', 'Views', 'Shares', 'content_type_encoded', 'region_encoded']
   Accuracy:        0.3611
   F1-Score:        0.3601


## Step 4: Build the Flask API
> Define all 5 endpoints for the DropQueen REST API

In [8]:
app = Flask(__name__)

# ── Label maps ────────────────────────────────────────────────────────────────
ENGAGEMENT_LABELS = {0: 'Low', 1: 'Medium', 2: 'High'}

# ── Endpoint 1: Health Check ──────────────────────────────────────────────────
@app.route('/health', methods=['GET'])
def health():
    """Returns API health status and model metadata."""
    return jsonify({
        'status'    : 'healthy',
        'api'       : 'DropQueen Demand & Trend Forecasting API',
        'version'   : '1.0.0',
        'timestamp' : datetime.utcnow().isoformat(),
        'models'    : {
            'demand_model'     : type(demand_model).__name__,
            'engagement_model' : type(engagement_model).__name__
        }
    }), 200


# ── Endpoint 2: Predict Demand Spike ─────────────────────────────────────────
@app.route('/predict/demand', methods=['POST'])
def predict_demand():
    """
    Predict whether a product will have a demand spike.

    Expected JSON body:
    {
        "product_id"       : "B08XYZ123",
        "weekly_reviews"   : 120,
        "avg_weekly_rating": 4.5,
        "rolling_avg_3w"   : 95.0,
        "week"             : 14,
        "high_rating_week" : 1,
        "product_encoded"  : 42
    }
    """
    try:
        data = request.get_json()

        if not data:
            return jsonify({'error': 'No JSON body provided'}), 400

        product_id = data.get('product_id', 'unknown')

        # Build week cyclical features
        week = data.get('week', 1)
        week_sin = np.sin(2 * np.pi * week / 52)
        week_cos = np.cos(2 * np.pi * week / 52)

        # Assemble feature vector
        feature_values = {
            'weekly_reviews'   : data.get('weekly_reviews', 0),
            'avg_weekly_rating': data.get('avg_weekly_rating', 0),
            'rolling_avg_3w'   : data.get('rolling_avg_3w', 0),
            'week_sin'         : week_sin,
            'week_cos'         : week_cos,
            'high_rating_week' : data.get('high_rating_week', 0),
            'product_encoded'  : data.get('product_encoded', 0)
        }

        X = pd.DataFrame([[feature_values[f] for f in DEMAND_FEATURES]],
                         columns=DEMAND_FEATURES)

        prediction    = int(demand_model.predict(X)[0])
        proba         = demand_model.predict_proba(X)[0]
        spike_prob    = round(float(proba[1]) if len(proba) > 1 else float(proba[0]), 4)
        no_spike_prob = round(1 - spike_prob, 4)

        recommendation = 'HIGH DEMAND EXPECTED — Stock up now!' if prediction == 1 \
                         else 'Normal demand — No spike predicted'

        return jsonify({
            'product_id'           : product_id,
            'demand_spike'         : bool(prediction),
            'spike_probability'    : spike_prob,
            'no_spike_probability' : no_spike_prob,
            'confidence'           : max(spike_prob, no_spike_prob),
            'recommendation'       : recommendation,
            'features_used'        : DEMAND_FEATURES,
            'timestamp'            : datetime.utcnow().isoformat()
        }), 200

    except Exception as e:
        return jsonify({'error': str(e)}), 500


# ── Endpoint 3: Predict TikTok Trend / Engagement ────────────────────────────
@app.route('/predict/trend', methods=['POST'])
def predict_trend():
    """
    Predict TikTok engagement level (Low / Medium / High).

    Expected JSON body:
    {
        "post_id"              : "Post_123",
        "views"                : 500000,
        "likes"                : 45000,
        "shares"               : 12000,
        "comments"             : 3000,
        "content_type_encoded" : 2,
        "region_encoded"       : 4
    }
    """
    try:
        data = request.get_json()

        if not data:
            return jsonify({'error': 'No JSON body provided'}), 400

        post_id = data.get('post_id', 'unknown')

        feature_map = {
            'Views'                : data.get('views', 0),
            'Likes'                : data.get('likes', 0),
            'Shares'               : data.get('shares', 0),
            'Comments'             : data.get('comments', 0),
            'content_type_encoded' : data.get('content_type_encoded', 0),
            'region_encoded'       : data.get('region_encoded', 0)
        }

        X = pd.DataFrame([[feature_map[f] for f in TIKTOK_FEATURES]],
                         columns=TIKTOK_FEATURES)

        prediction     = int(engagement_model.predict(X)[0])
        label          = ENGAGEMENT_LABELS.get(prediction, 'Unknown')
        proba          = engagement_model.predict_proba(X)[0]
        confidence     = round(float(max(proba)), 4)

        trending = label == 'High'
        recommendation = 'TRENDING — High engagement predicted! Great time to stock up.' \
                         if trending else \
                         f'{label} engagement predicted — Monitor for changes.'

        return jsonify({
            'post_id'          : post_id,
            'engagement_level' : label,
            'is_trending'      : trending,
            'confidence'       : confidence,
            'probabilities'    : {
                'Low'    : round(float(proba[0]), 4),
                'Medium' : round(float(proba[1]), 4),
                'High'   : round(float(proba[2]), 4)
            },
            'recommendation'   : recommendation,
            'timestamp'        : datetime.utcnow().isoformat()
        }), 200

    except Exception as e:
        return jsonify({'error': str(e)}), 500


# ── Endpoint 4: Top Predicted Products ───────────────────────────────────────
@app.route('/products/top', methods=['GET'])
def top_products():
    """
    Returns a simulated list of top predicted demand products.
    In production this would query live Amazon/TikTok data.
    """
    sample_products = [
        {'rank': 1, 'product_id': 'B08XYZ111', 'category': 'Skincare',
         'predicted_spike': True,  'confidence': 0.96,
         'trend_signal': 'TikTok viral — #NiacinamideSerum',
         'recommended_action': 'Restock immediately'},
        {'rank': 2, 'product_id': 'B08XYZ222', 'category': 'Haircare',
         'predicted_spike': True,  'confidence': 0.91,
         'trend_signal': 'Amazon rank climbing fast',
         'recommended_action': 'Increase inventory by 30%'},
        {'rank': 3, 'product_id': 'B08XYZ333', 'category': 'Makeup',
         'predicted_spike': True,  'confidence': 0.87,
         'trend_signal': 'Influencer feature detected',
         'recommended_action': 'Prepare for surge'},
        {'rank': 4, 'product_id': 'B08XYZ444', 'category': 'Fragrance',
         'predicted_spike': False, 'confidence': 0.74,
         'trend_signal': 'Moderate social mention growth',
         'recommended_action': 'Monitor closely'},
        {'rank': 5, 'product_id': 'B08XYZ555', 'category': 'Bodycare',
         'predicted_spike': True,  'confidence': 0.82,
         'trend_signal': '#SelfCare TikTok trend surge',
         'recommended_action': 'Restock within 48 hours'}
    ]

    return jsonify({
        'top_products'  : sample_products,
        'total_tracked' : 10000,
        'platforms'     : ['TikTok Shop', 'Amazon'],
        'generated_at'  : datetime.utcnow().isoformat()
    }), 200


# ── Endpoint 5: Model Versions ────────────────────────────────────────────────
@app.route('/models/versions', methods=['GET'])
def model_versions():
    """Returns metadata about the currently deployed models."""
    return jsonify({
        'models': [
            {
                'name'         : 'demand_spike_model',
                'type'         : type(demand_model).__name__,
                'version'      : 'v1.0',
                'accuracy'     : model_config['demand_accuracy'],
                'f1_score'     : model_config['demand_f1'],
                'features'     : DEMAND_FEATURES,
                'target'       : 'demand_spike (0 = no spike, 1 = spike)',
                'trained_on'   : 'Amazon Beauty Reviews Dataset'
            },
            {
                'name'         : 'tiktok_engagement_model',
                'type'         : type(engagement_model).__name__,
                'version'      : 'v1.0',
                'accuracy'     : model_config['tiktok_accuracy'],
                'f1_score'     : model_config['tiktok_f1'],
                'features'     : TIKTOK_FEATURES,
                'target'       : 'engagement_level (0=Low, 1=Medium, 2=High)',
                'trained_on'   : 'TikTok Trending Products Dataset'
            }
        ],
        'api_version' : '1.0.0',
        'timestamp'   : datetime.utcnow().isoformat()
    }), 200


print('✅ Flask app defined with 5 endpoints:')
print('   GET  /health')
print('   POST /predict/demand')
print('   POST /predict/trend')
print('   GET  /products/top')
print('   GET  /models/versions')

✅ Flask app defined with 5 endpoints:
   GET  /health
   POST /predict/demand
   POST /predict/trend
   GET  /products/top
   GET  /models/versions


## Step 5: Launch the API with ngrok
> ngrok creates a public URL so we can test the API from inside Colab

In [11]:
import threading
import time

# ── Kill any previous Flask instances ────────────────────────────────────
import os
os.system("kill -9 $(lsof -t -i:5000) 2>/dev/null || true")
os.system("kill -9 $(lsof -t -i:5001) 2>/dev/null || true")
time.sleep(2)

# ── Start Flask in background thread ─────────────────────────────────────
def run_flask():
    app.run(port=5001, use_reloader=False, threaded=True)

flask_thread = threading.Thread(target=run_flask, daemon=True)
flask_thread.start()
time.sleep(3)

BASE_URL = "http://127.0.0.1:5001"

print('🚀 DropQueen API is LIVE!')
print(f'   Local URL: {BASE_URL}')
print(f'\n   Endpoints:')
print(f'   {BASE_URL}/health')
print(f'   {BASE_URL}/predict/demand')
print(f'   {BASE_URL}/predict/trend')
print(f'   {BASE_URL}/products/top')
print(f'   {BASE_URL}/models/versions')

 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5001
INFO:werkzeug:Press CTRL+C to quit


🚀 DropQueen API is LIVE!
   Local URL: http://127.0.0.1:5001

   Endpoints:
   http://127.0.0.1:5001/health
   http://127.0.0.1:5001/predict/demand
   http://127.0.0.1:5001/predict/trend
   http://127.0.0.1:5001/products/top
   http://127.0.0.1:5001/models/versions


## Step 6: Test All API Endpoints
> Run each cell to verify every endpoint works correctly

In [12]:
# ── Test 1: Health Check ──────────────────────────────────────────────────────
print('=' * 55)
print('TEST 1: GET /health')
print('=' * 55)

resp = requests.get(f'{BASE_URL}/health')
print(f'Status Code: {resp.status_code}')
print(json.dumps(resp.json(), indent=2))

INFO:werkzeug:127.0.0.1 - - [06/Mar/2026 22:20:49] "GET /health HTTP/1.1" 200 -


TEST 1: GET /health
Status Code: 200
{
  "api": "DropQueen Demand & Trend Forecasting API",
  "models": {
    "demand_model": "RandomForestClassifier",
    "engagement_model": "LogisticRegression"
  },
  "status": "healthy",
  "timestamp": "2026-03-06T22:20:49.463321",
  "version": "1.0.0"
}


In [13]:
# ── Test 2: Predict Demand ────────────────────────────────────────────────────
print('=' * 55)
print('TEST 2: POST /predict/demand')
print('=' * 55)

demand_payload = {
    'product_id'        : 'B08XYZ123',
    'weekly_reviews'    : 120,
    'avg_weekly_rating' : 4.5,
    'rolling_avg_3w'    : 95.0,
    'week'              : 14,
    'high_rating_week'  : 1,
    'product_encoded'   : 42
}

resp = requests.post(f'{BASE_URL}/predict/demand', json=demand_payload)
print(f'Status Code: {resp.status_code}')
print(json.dumps(resp.json(), indent=2))

INFO:werkzeug:127.0.0.1 - - [06/Mar/2026 22:20:54] "POST /predict/demand HTTP/1.1" 200 -


TEST 2: POST /predict/demand
Status Code: 200
{
  "confidence": 1.0,
  "demand_spike": true,
  "features_used": [
    "weekly_reviews",
    "avg_weekly_rating",
    "rolling_avg_3w",
    "week_sin",
    "week_cos",
    "high_rating_week",
    "product_encoded"
  ],
  "no_spike_probability": 0.0,
  "product_id": "B08XYZ123",
  "recommendation": "HIGH DEMAND EXPECTED \u2014 Stock up now!",
  "spike_probability": 1.0,
  "timestamp": "2026-03-06T22:20:54.413834"
}


In [14]:
# ── Test 3: Predict Trend ─────────────────────────────────────────────────────
print('=' * 55)
print('TEST 3: POST /predict/trend')
print('=' * 55)

trend_payload = {
    'post_id'              : 'Post_456',
    'views'                : 3500000,
    'likes'                : 280000,
    'shares'               : 65000,
    'comments'             : 18000,
    'content_type_encoded' : 2,
    'region_encoded'       : 4
}

resp = requests.post(f'{BASE_URL}/predict/trend', json=trend_payload)
print(f'Status Code: {resp.status_code}')
print(json.dumps(resp.json(), indent=2))

INFO:werkzeug:127.0.0.1 - - [06/Mar/2026 22:20:56] "POST /predict/trend HTTP/1.1" 200 -


TEST 3: POST /predict/trend
Status Code: 200
{
  "confidence": 0.3355,
  "engagement_level": "Low",
  "is_trending": false,
  "post_id": "Post_456",
  "probabilities": {
    "High": 0.3329,
    "Low": 0.3355,
    "Medium": 0.3316
  },
  "recommendation": "Low engagement predicted \u2014 Monitor for changes.",
  "timestamp": "2026-03-06T22:20:56.922693"
}


In [15]:
# ── Test 4: Top Products ──────────────────────────────────────────────────────
print('=' * 55)
print('TEST 4: GET /products/top')
print('=' * 55)

resp = requests.get(f'{BASE_URL}/products/top')
print(f'Status Code: {resp.status_code}')
print(json.dumps(resp.json(), indent=2))

INFO:werkzeug:127.0.0.1 - - [06/Mar/2026 22:20:59] "GET /products/top HTTP/1.1" 200 -


TEST 4: GET /products/top
Status Code: 200
{
  "generated_at": "2026-03-06T22:20:59.172287",
  "platforms": [
    "TikTok Shop",
    "Amazon"
  ],
  "top_products": [
    {
      "category": "Skincare",
      "confidence": 0.96,
      "predicted_spike": true,
      "product_id": "B08XYZ111",
      "rank": 1,
      "recommended_action": "Restock immediately",
      "trend_signal": "TikTok viral \u2014 #NiacinamideSerum"
    },
    {
      "category": "Haircare",
      "confidence": 0.91,
      "predicted_spike": true,
      "product_id": "B08XYZ222",
      "rank": 2,
      "recommended_action": "Increase inventory by 30%",
      "trend_signal": "Amazon rank climbing fast"
    },
    {
      "category": "Makeup",
      "confidence": 0.87,
      "predicted_spike": true,
      "product_id": "B08XYZ333",
      "rank": 3,
      "recommended_action": "Prepare for surge",
      "trend_signal": "Influencer feature detected"
    },
    {
      "category": "Fragrance",
      "confidence": 0.74,
 

In [16]:
# ── Test 5: Model Versions ────────────────────────────────────────────────────
print('=' * 55)
print('TEST 5: GET /models/versions')
print('=' * 55)

resp = requests.get(f'{BASE_URL}/models/versions')
print(f'Status Code: {resp.status_code}')
print(json.dumps(resp.json(), indent=2))

INFO:werkzeug:127.0.0.1 - - [06/Mar/2026 22:21:02] "GET /models/versions HTTP/1.1" 200 -


TEST 5: GET /models/versions
Status Code: 200
{
  "api_version": "1.0.0",
  "models": [
    {
      "accuracy": 1.0,
      "f1_score": 1.0,
      "features": [
        "weekly_reviews",
        "avg_weekly_rating",
        "rolling_avg_3w",
        "week_sin",
        "week_cos",
        "high_rating_week",
        "product_encoded"
      ],
      "name": "demand_spike_model",
      "target": "demand_spike (0 = no spike, 1 = spike)",
      "trained_on": "Amazon Beauty Reviews Dataset",
      "type": "RandomForestClassifier",
      "version": "v1.0"
    },
    {
      "accuracy": 0.3611,
      "f1_score": 0.3601,
      "features": [
        "Likes",
        "Comments",
        "Views",
        "Shares",
        "content_type_encoded",
        "region_encoded"
      ],
      "name": "tiktok_engagement_model",
      "target": "engagement_level (0=Low, 1=Medium, 2=High)",
      "trained_on": "TikTok Trending Products Dataset",
      "type": "LogisticRegression",
      "version": "v1.0"
   

## Step 7: Save app.py for Docker Deployment
> Export the full Flask app as `app.py` — ready to be containerized with Docker

In [17]:
app_py = '''
"""
👑 DropQueen — Flask REST API
AI-Powered Product Demand & Sales Forecasting Engine
For TikTok Shop & Amazon

Author : Chastity Lewis
Course : CISC 610 — DevOps and MLOps | Mercy University | Spring 2026
"""

import os
import json
import joblib
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

from flask import Flask, request, jsonify
from datetime import datetime

app = Flask(__name__)

# ── Load Models ────────────────────────────────────────────────────────────────
demand_model     = joblib.load("demand_model.pkl")
engagement_model = joblib.load("engagement_model.pkl")

with open("model_config.json", "r") as f:
    model_config = json.load(f)

DEMAND_FEATURES = model_config["demand_features"]
TIKTOK_FEATURES = model_config["tiktok_features"]
ENGAGEMENT_LABELS = {0: "Low", 1: "Medium", 2: "High"}

# ── Endpoint 1: Health Check ──────────────────────────────────────────────────
@app.route("/health", methods=["GET"])
def health():
    return jsonify({
        "status"    : "healthy",
        "api"       : "DropQueen Demand & Trend Forecasting API",
        "version"   : "1.0.0",
        "timestamp" : datetime.utcnow().isoformat(),
        "models"    : {
            "demand_model"     : type(demand_model).__name__,
            "engagement_model" : type(engagement_model).__name__
        }
    }), 200


# ── Endpoint 2: Predict Demand Spike ─────────────────────────────────────────
@app.route("/predict/demand", methods=["POST"])
def predict_demand():
    try:
        data = request.get_json()
        if not data:
            return jsonify({"error": "No JSON body provided"}), 400

        product_id = data.get("product_id", "unknown")
        week       = data.get("week", 1)
        week_sin   = np.sin(2 * np.pi * week / 52)
        week_cos   = np.cos(2 * np.pi * week / 52)

        feature_values = {
            "weekly_reviews"   : data.get("weekly_reviews", 0),
            "avg_weekly_rating": data.get("avg_weekly_rating", 0),
            "rolling_avg_3w"   : data.get("rolling_avg_3w", 0),
            "week_sin"         : week_sin,
            "week_cos"         : week_cos,
            "high_rating_week" : data.get("high_rating_week", 0),
            "product_encoded"  : data.get("product_encoded", 0)
        }

        X = pd.DataFrame([[feature_values[f] for f in DEMAND_FEATURES]],
                         columns=DEMAND_FEATURES)

        prediction    = int(demand_model.predict(X)[0])
        proba         = demand_model.predict_proba(X)[0]
        spike_prob    = round(float(proba[1]) if len(proba) > 1 else float(proba[0]), 4)
        no_spike_prob = round(1 - spike_prob, 4)

        recommendation = "HIGH DEMAND EXPECTED — Stock up now!" if prediction == 1 \
                         else "Normal demand — No spike predicted"

        return jsonify({
            "product_id"           : product_id,
            "demand_spike"         : bool(prediction),
            "spike_probability"    : spike_prob,
            "no_spike_probability" : no_spike_prob,
            "confidence"           : max(spike_prob, no_spike_prob),
            "recommendation"       : recommendation,
            "features_used"        : DEMAND_FEATURES,
            "timestamp"            : datetime.utcnow().isoformat()
        }), 200

    except Exception as e:
        return jsonify({"error": str(e)}), 500


# ── Endpoint 3: Predict TikTok Trend ─────────────────────────────────────────
@app.route("/predict/trend", methods=["POST"])
def predict_trend():
    try:
        data = request.get_json()
        if not data:
            return jsonify({"error": "No JSON body provided"}), 400

        post_id = data.get("post_id", "unknown")

        feature_map = {
            "Views"                : data.get("views", 0),
            "Likes"                : data.get("likes", 0),
            "Shares"               : data.get("shares", 0),
            "Comments"             : data.get("comments", 0),
            "content_type_encoded" : data.get("content_type_encoded", 0),
            "region_encoded"       : data.get("region_encoded", 0)
        }

        X = pd.DataFrame([[feature_map[f] for f in TIKTOK_FEATURES]],
                         columns=TIKTOK_FEATURES)

        prediction = int(engagement_model.predict(X)[0])
        label      = ENGAGEMENT_LABELS.get(prediction, "Unknown")
        proba      = engagement_model.predict_proba(X)[0]
        confidence = round(float(max(proba)), 4)
        trending   = label == "High"

        recommendation = "TRENDING — High engagement predicted! Great time to stock up." \
                         if trending else f"{label} engagement predicted — Monitor for changes."

        return jsonify({
            "post_id"          : post_id,
            "engagement_level" : label,
            "is_trending"      : trending,
            "confidence"       : confidence,
            "probabilities"    : {
                "Low"    : round(float(proba[0]), 4),
                "Medium" : round(float(proba[1]), 4),
                "High"   : round(float(proba[2]), 4)
            },
            "recommendation"   : recommendation,
            "timestamp"        : datetime.utcnow().isoformat()
        }), 200

    except Exception as e:
        return jsonify({"error": str(e)}), 500


# ── Endpoint 4: Top Products ──────────────────────────────────────────────────
@app.route("/products/top", methods=["GET"])
def top_products():
    sample_products = [
        {"rank": 1, "product_id": "B08XYZ111", "category": "Skincare",
         "predicted_spike": True,  "confidence": 0.96,
         "trend_signal": "TikTok viral — #NiacinamideSerum",
         "recommended_action": "Restock immediately"},
        {"rank": 2, "product_id": "B08XYZ222", "category": "Haircare",
         "predicted_spike": True,  "confidence": 0.91,
         "trend_signal": "Amazon rank climbing fast",
         "recommended_action": "Increase inventory by 30%"},
        {"rank": 3, "product_id": "B08XYZ333", "category": "Makeup",
         "predicted_spike": True,  "confidence": 0.87,
         "trend_signal": "Influencer feature detected",
         "recommended_action": "Prepare for surge"},
        {"rank": 4, "product_id": "B08XYZ444", "category": "Fragrance",
         "predicted_spike": False, "confidence": 0.74,
         "trend_signal": "Moderate social mention growth",
         "recommended_action": "Monitor closely"},
        {"rank": 5, "product_id": "B08XYZ555", "category": "Bodycare",
         "predicted_spike": True,  "confidence": 0.82,
         "trend_signal": "#SelfCare TikTok trend surge",
         "recommended_action": "Restock within 48 hours"}
    ]
    return jsonify({
        "top_products"  : sample_products,
        "total_tracked" : 10000,
        "platforms"     : ["TikTok Shop", "Amazon"],
        "generated_at"  : datetime.utcnow().isoformat()
    }), 200


# ── Endpoint 5: Model Versions ────────────────────────────────────────────────
@app.route("/models/versions", methods=["GET"])
def model_versions():
    return jsonify({
        "models": [
            {
                "name"       : "demand_spike_model",
                "type"       : type(demand_model).__name__,
                "version"    : "v1.0",
                "accuracy"   : model_config["demand_accuracy"],
                "f1_score"   : model_config["demand_f1"],
                "features"   : DEMAND_FEATURES,
                "target"     : "demand_spike (0=no spike, 1=spike)",
                "trained_on" : "Amazon Beauty Reviews Dataset"
            },
            {
                "name"       : "tiktok_engagement_model",
                "type"       : type(engagement_model).__name__,
                "version"    : "v1.0",
                "accuracy"   : model_config["tiktok_accuracy"],
                "f1_score"   : model_config["tiktok_f1"],
                "features"   : TIKTOK_FEATURES,
                "target"     : "engagement_level (0=Low, 1=Medium, 2=High)",
                "trained_on" : "TikTok Trending Products Dataset"
            }
        ],
        "api_version" : "1.0.0",
        "timestamp"   : datetime.utcnow().isoformat()
    }), 200


# ── Run ───────────────────────────────────────────────────────────────────────
if __name__ == "__main__":
    port = int(os.environ.get("PORT", 5000))
    app.run(host="0.0.0.0", port=port, debug=False)
'''

with open('app.py', 'w') as f:
    f.write(app_py)

print('✅ app.py saved — ready for Docker deployment!')

# Download it
files.download('app.py')
print('📥 app.py downloaded!')

✅ app.py saved — ready for Docker deployment!


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

📥 app.py downloaded!


## Step 8: Notebook 4 Summary

In [18]:
print('=' * 60)
print('        👑 DropQueen — Notebook 4 Summary')
print('=' * 60)
print()
print('🚀 FLASK API — 5 Endpoints Deployed:')
print('   GET  /health            ✅ Returns API health status')
print('   POST /predict/demand    ✅ Predicts product demand spike')
print('   POST /predict/trend     ✅ Predicts TikTok engagement level')
print('   GET  /products/top      ✅ Returns top predicted drops')
print('   GET  /models/versions   ✅ Lists model metadata')
print()
print('💾 Output Files:')
print('   app.py   ← Flask API (ready for Docker)')
print()
print('🌲 Demand Model Accuracy  :', model_config["demand_accuracy"])
print('📊 Engagement Model Accuracy:', model_config["tiktok_accuracy"])
print()
print('=' * 60)
print()
print('✅ Next: Notebook 05 — Docker & CI/CD Deployment')

        👑 DropQueen — Notebook 4 Summary

🚀 FLASK API — 5 Endpoints Deployed:
   GET  /health            ✅ Returns API health status
   POST /predict/demand    ✅ Predicts product demand spike
   POST /predict/trend     ✅ Predicts TikTok engagement level
   GET  /products/top      ✅ Returns top predicted drops
   GET  /models/versions   ✅ Lists model metadata

💾 Output Files:
   app.py   ← Flask API (ready for Docker)

🌲 Demand Model Accuracy  : 1.0
📊 Engagement Model Accuracy: 0.3611


✅ Next: Notebook 05 — Docker & CI/CD Deployment
